# Notebook 01 — Data Pipeline

**Across the Water: French Nuclear Unplanned Outages and GB Power Prices**

Fetches all raw data, caches to `data/` as parquet, and estimates the
reduced-form IFA flow regression used for position sizing throughout the project.

**Run this notebook first.** All subsequent notebooks depend on the parquet files produced here.

---

### Outputs

| File | Description |
|---|---|
| `entso_unavailability.parquet` | FR nuclear unavailability notices — fetch timestamps logged (backtest integrity) |
| `elexon_da_prices.parquet` | GB DA baseload prices, daily volume-weighted (Elexon MID) |
| `elexon_mid_halfhourly.parquet` | GB intraday prices, half-hourly (Elexon MID — proxy for EPEX GB ID) |
| `neso_historic_demand.parquet` | NESO half-hourly demand, wind, solar, IFA1/IFA2 flows |
| `rte_generation.parquet` | RTE éCO2mix FR generation mix + nuclear forecast |
| `fr_temperature.parquet` | FR daily temperature + seasonal deviation (open-meteo) |
| `ttf_spot.parquet` | TTF natural gas spot price (EMBER) |
| `de_wind_generation.parquet` | DE wind generation onshore + offshore (ENTSO-E A75) |
| `fr_da_price.parquet` | FR day-ahead electricity price EUR/MWh (ENTSO-E A44) |
| `ifa_flow_model.parquet` | Reduced-form IFA1/IFA2 flow regression coefficients |

---

### Setup

Create a `.env` file in the project root with your ENTSO-E API key:
```
ENTSO_API_KEY=your_key_here
```
Free registration: https://transparency.entsoe.eu

To force a re-fetch of any dataset, delete the corresponding `.parquet` file:
```python
(DATA_DIR / "dataset_name.parquet").unlink(missing_ok=True)
```

> ✅ **COMPLETE** — All data fetched and cached. IFA flow regression estimated.
> Proceed to Notebook 02.

---
## 1. Imports

In [1]:
import sys, os
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import subprocess
for pkg in ["python-dotenv", "statsmodels", "lightgbm", "seaborn", "yfinance"]:
    try:
        __import__(pkg.replace("-", "_").split(".")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm

try:
    from dotenv import load_dotenv
    load_dotenv(project_root / ".env")
except ImportError:
    pass

from src import (
    log, DATA_DIR, save, load,
    SAMPLE_START, MAIN_START, MAIN_END,
    fetch_entso_unavailability,
    fetch_elexon_da_prices,
    fetch_elexon_mid_halfhourly,
    fetch_neso_historic_demand,
    fetch_rte_generation,
    fetch_fr_temperature,
    fetch_ttf_spot,
    fetch_de_wind_generation,
    fetch_fr_da_price,
)

ENTSO_KEY = os.getenv("ENTSO_API_KEY")
if not ENTSO_KEY:
    print("⚠️  ENTSO_API_KEY not set — ENTSO-E datasets will be skipped.")
    print("   Add ENTSO_API_KEY=your_key to .env in the project root.")

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.2f}".format)
print(f"DATA_DIR : {DATA_DIR}")
print(f"Period   : {SAMPLE_START} → {MAIN_END}")

DATA_DIR : /home/ndrew/across-the-water/data
Period   : 2018-01-01 → 2026-04-01


---
## 2. Helpers

In [2]:
def fetch_or_load(name: str, fetch_fn, *args, **kwargs) -> pd.DataFrame:
    """
    Load from parquet cache if present, otherwise fetch and save.
    Delete the parquet file to force a re-fetch.
    """
    path = DATA_DIR / f"{name}.parquet"
    if path.exists():
        log.info("%-35s loading from cache", name)
        return pd.read_parquet(path)
    log.info("%-35s no cache — fetching...", name)
    df = fetch_fn(*args, **kwargs)
    if not df.empty:
        df.to_parquet(path)
        log.info("%-35s saved (%s rows)", name, f"{len(df):,}")
    else:
        log.warning("%-35s fetch returned empty — not cached", name)
    return df


def safe_head(df: pd.DataFrame, cols: list, n: int = 5) -> pd.DataFrame:
    """
    Show head of df using only columns that actually exist.
    Prints a note for any requested columns that are absent.
    Prevents KeyErrors when the API returns a different schema than expected.
    """
    if df.empty:
        print("DataFrame is empty.")
        return df
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"Note: columns not present in this dataset: {missing}")
    return df[present].head(n) if present else df.head(n)

---
## 3. Fetch All Datasets

Each call is idempotent — cached results load in under a second.

**ENTSO-E datasets** require `ENTSO_API_KEY` and take ~10 min each on first run
(monthly chunks, rate-limited). All others are fast.

### 3.1 ENTSO-E — FR Nuclear Unavailability

Unit-level planned and unplanned outage notices. Fetch timestamps are logged
as a backtest integrity constraint — notices are sometimes filed retroactively.

In [3]:
if ENTSO_KEY:
    entso_df = fetch_or_load(
        "entso_unavailability",
        fetch_entso_unavailability,
        api_key=ENTSO_KEY, start=SAMPLE_START, end=MAIN_END,
    )
else:
    print("Skipping — no ENTSO_API_KEY.")
    entso_df = pd.DataFrame()

safe_head(entso_df, ["asset_name", "unit_mw", "unavailable_mw", "outage_type", "fetch_timestamp"])

2026-04-09 08:49:51  INFO      entso_unavailability                loading from cache


,asset_name,unit_mw,unavailable_mw,outage_type,fetch_timestamp
datetime_utc,,,,,
2017-04-21 09:30:00+00:00,REVIN,808.00,476.28,unplanned,2026-04-08T16:38:55.602273+00:00
2017-04-21 09:30:00+00:00,REVIN,808.00,476.28,unplanned,2026-04-08T16:38:55.602273+00:00
2017-04-21 09:30:00+00:00,REVIN,808.00,476.28,unplanned,2026-04-08T16:38:55.602273+00:00
2017-04-21 09:30:00+00:00,REVIN,808.00,476.28,unplanned,2026-04-08T16:38:55.602273+00:00
2017-04-21 09:30:00+00:00,REVIN,808.00,476.28,unplanned,2026-04-08T16:38:55.602273+00:00


### 3.2 Elexon — GB Day-Ahead Prices (daily)

Volume-weighted daily average from Elexon MID (N2EX + APXMIDP providers).

In [4]:
elexon_df = fetch_or_load("elexon_da_prices", fetch_elexon_da_prices)
safe_head(elexon_df, ["gb_da_price", "fetch_timestamp"])

2026-04-09 08:49:51  INFO      elexon_da_prices                    loading from cache


,gb_da_price,fetch_timestamp
date,,
2018-01-01 00:00:00+00:00,49.66,2026-04-08T15:16:10.926754+00:00
2018-01-02 00:00:00+00:00,50.60,2026-04-08T15:16:10.926754+00:00
2018-01-03 00:00:00+00:00,48.89,2026-04-08T15:16:10.926754+00:00
2018-01-04 00:00:00+00:00,46.61,2026-04-08T15:16:10.926754+00:00
2018-01-05 00:00:00+00:00,57.84,2026-04-08T15:16:10.926754+00:00


### 3.3 Elexon — GB Intraday Prices (half-hourly)

Elexon MID at half-hourly resolution — intraday price proxy for Notebook 02b.
EPEX GB hourly ID settlement prices are not publicly available post-Brexit;
MID (N2EX/APXMIDP volume-weighted) is the closest free equivalent.

**Test window:** settlement period 33 = 16:00–16:30, period 34 = 16:30–17:00.
Primary 02b test window (first bar entirely post-announcement) = periods 33–34.

In [5]:
mid_hh_df = fetch_or_load("elexon_mid_halfhourly", fetch_elexon_mid_halfhourly)
safe_head(mid_hh_df, ["gb_id_price_gbp_mwh", "gb_id_volume_mwh", "fetch_timestamp"])

2026-04-09 08:49:51  INFO      elexon_mid_halfhourly               loading from cache


,gb_id_price_gbp_mwh,gb_id_volume_mwh,fetch_timestamp
datetime_utc,,,
2018-01-01 00:00:00+00:00,47.27,781.35,2026-04-08T17:04:08.499393+00:00
2018-01-01 00:30:00+00:00,48.57,655.40,2026-04-08T17:04:08.499393+00:00
2018-01-01 01:00:00+00:00,52.32,821.50,2026-04-08T17:04:08.499393+00:00
2018-01-01 01:30:00+00:00,51.18,815.20,2026-04-08T17:04:08.499393+00:00
2018-01-01 02:00:00+00:00,46.08,709.05,2026-04-08T17:04:08.499393+00:00


### 3.4 NESO — Historic Demand, Wind, IFA Flows

Half-hourly: GB demand, embedded wind/solar, IFA1 and IFA2 interconnector flows.
One CSV per year; direct download (SQL datastore returns 409).

In [6]:
neso_df = fetch_or_load("neso_historic_demand", fetch_neso_historic_demand)
safe_head(neso_df, ["demand_actual_mw", "wind_actual_mw", "ifa1_flow_mw", "ifa2_flow_mw"])

2026-04-09 08:49:51  INFO      neso_historic_demand                loading from cache


,demand_actual_mw,wind_actual_mw,ifa1_flow_mw,ifa2_flow_mw
datetime_utc,,,,
2018-01-01 00:00:00+00:00,25593,3064,1415,0
2018-01-01 00:30:00+00:00,26349,3005,1404,0
2018-01-01 01:00:00+00:00,26240,2950,1405,0
2018-01-01 01:30:00+00:00,25292,2895,1405,0
2018-01-01 02:00:00+00:00,24416,2892,1308,0


### 3.5 RTE éCO2mix — FR Generation Mix

Half-hourly French generation by type: nuclear actual + forecast, exports, imports.
Used for the reduced-form IFA flow regression and as SCM regression controls.

Note: `fr_nuclear_forecast_mw` is sparsely populated by the RTE API and may
be absent from older cached data — this is expected and does not affect the
IFA regression, which uses `fr_nuclear_actual_mw` only.

In [7]:
rte_df = fetch_or_load("rte_generation", fetch_rte_generation)
safe_head(rte_df, ["fr_nuclear_actual_mw", "fr_nuclear_forecast_mw", "fr_net_exports_mw", "fr_consumption_mw"])

2026-04-09 08:49:51  INFO      rte_generation                      loading from cache


Note: columns not present in this dataset: ['fr_nuclear_forecast_mw']


,fr_nuclear_actual_mw,fr_net_exports_mw,fr_consumption_mw
datetime_utc,,,
2018-01-01 00:00:00+00:00,35725.00,590.00,57879.00
2018-01-01 00:15:00+00:00,NaN,NaN,NaN
2018-01-01 00:30:00+00:00,35182.00,1538.00,57901.00
2018-01-01 00:45:00+00:00,NaN,NaN,NaN
2018-01-01 01:00:00+00:00,34914.00,1539.00,57261.00


### 3.6 FR Temperature

Daily mean temperature at the French centroid (open-meteo archive API).
Seasonal deviation used as a demand confounder in the SCM regression.

In [8]:
temp_df = fetch_or_load("fr_temperature", fetch_fr_temperature)
safe_head(temp_df, ["fr_temp_c", "fr_temp_monthly_mean", "fr_temp_deviation"])

2026-04-09 08:49:52  INFO      fr_temperature                      loading from cache


,fr_temp_c,fr_temp_monthly_mean,fr_temp_deviation
date,,,
2018-01-01 00:00:00+00:00,6.50,6.50,0.00
2018-01-02 00:00:00+00:00,7.90,7.20,0.70
2018-01-03 00:00:00+00:00,10.50,8.30,2.20
2018-01-04 00:00:00+00:00,11.80,9.18,2.62
2018-01-05 00:00:00+00:00,9.70,9.28,0.42


### 3.7 TTF Natural Gas Spot

Daily EUR/MWh close price from Yahoo Finance (TTF=F — Dutch TTF front-month futures).
Cost confounder: TTF shocks affect both French generation costs and GB marginal cost
simultaneously. Weekend/holiday gaps forward-filled by 1 day (prices do not change
on non-trading days).

In [9]:
ttf_df = fetch_or_load("ttf_spot", fetch_ttf_spot)
safe_head(ttf_df, ["ttf_spot_eur_mwh", "ttf_is_d1_fallback"])

2026-04-09 08:49:52  INFO      ttf_spot                            loading from cache


Price,ttf_spot_eur_mwh,ttf_is_d1_fallback
date,,
2018-01-02 00:00:00+00:00,19.32,False
2018-01-03 00:00:00+00:00,19.33,False
2018-01-04 00:00:00+00:00,19.20,False
2018-01-05 00:00:00+00:00,18.92,False
2018-01-06 00:00:00+00:00,18.92,False


### 3.8 ENTSO-E — German Wind Generation

Hourly onshore (B19) + offshore (B18) actual generation for Germany.
Continental renewable proxy in the SCM regression — high German wind
suppresses continental wholesale prices and reduces FR export incentives.

In [10]:
if ENTSO_KEY:
    de_wind_df = fetch_or_load(
        "de_wind_generation",
        fetch_de_wind_generation,
        api_key=ENTSO_KEY,
    )
else:
    print("Skipping — no ENTSO_API_KEY.")
    de_wind_df = pd.DataFrame()

safe_head(de_wind_df, ["de_wind_onshore_mw", "de_wind_offshore_mw", "de_wind_mw"])

2026-04-09 08:49:52  INFO      de_wind_generation                  loading from cache


,de_wind_onshore_mw,de_wind_offshore_mw,de_wind_mw
datetime_utc,,,
2018-01-01 00:00:00+00:00,29700.32,2997.92,32698.24
2018-01-01 01:00:00+00:00,30187.18,3093.07,33280.25
2018-01-01 02:00:00+00:00,31068.88,3134.40,34203.28
2018-01-01 03:00:00+00:00,31057.87,3209.39,34267.26
2018-01-01 04:00:00+00:00,31477.95,3278.63,34756.58


### 3.9 ENTSO-E — FR Day-Ahead Price

Hourly EUR/MWh from ENTSO-E A44 (Publication document).
Used in Model B only — conditioning on the FR DA clear at 08:00 isolates
the residual GB DA mispricing in the 08:00–11:00 window (IFA1 channel).

In [11]:
if ENTSO_KEY:
    fr_da_df = fetch_or_load(
        "fr_da_price",
        fetch_fr_da_price,
        api_key=ENTSO_KEY,
    )
else:
    print("Skipping — no ENTSO_API_KEY.")
    fr_da_df = pd.DataFrame()

safe_head(fr_da_df, ["fr_da_price_eur_mwh", "fetch_timestamp"])

2026-04-09 08:49:52  INFO      fr_da_price                         loading from cache


,fr_da_price_eur_mwh,fetch_timestamp
datetime_utc,,
2017-12-31 23:00:00+00:00,6.74,2026-04-08T16:49:32.126024+00:00
2018-01-01 00:00:00+00:00,4.74,2026-04-08T16:49:32.126024+00:00
2018-01-01 01:00:00+00:00,3.66,2026-04-08T16:49:32.126024+00:00
2018-01-01 02:00:00+00:00,1.26,2026-04-08T16:49:32.126024+00:00
2018-01-01 03:00:00+00:00,-20.10,2026-04-08T16:49:32.126024+00:00


---
## 4. Pipeline Summary

In [12]:
# Key columns to check for missing data — avoids inflated missing % from sparse ancillary fields
KEY_COLS = {
    "entso_unavailability":  ["unavailable_mw", "outage_type"],
    "elexon_da_prices":      ["gb_da_price"],
    "elexon_mid_halfhourly": ["gb_id_price_gbp_mwh", "gb_id_volume_mwh"],
    "neso_historic_demand":  ["demand_actual_mw", "ifa1_flow_mw", "ifa2_flow_mw"],
    "rte_generation":        ["fr_nuclear_actual_mw", "fr_net_exports_mw"],
    "fr_temperature":        ["fr_temp_c", "fr_temp_deviation"],
    "ttf_spot":              ["ttf_spot_eur_mwh"],
    "de_wind_generation":    ["de_wind_mw"],
    "fr_da_price":           ["fr_da_price_eur_mwh"],
}

datasets = {
    "entso_unavailability":  entso_df,
    "elexon_da_prices":      elexon_df,
    "elexon_mid_halfhourly": mid_hh_df,
    "neso_historic_demand":  neso_df,
    "rte_generation":        rte_df,
    "fr_temperature":        temp_df,
    "ttf_spot":              ttf_df,
    "de_wind_generation":    de_wind_df,
    "fr_da_price":           fr_da_df,
}

print(f"{'Dataset':<35} {'Rows':>8}  {'From':<12}  {'To':<12}  {'Missing':>8}  Status")
print("-" * 90)
for name, df in datasets.items():
    if df.empty:
        print(f"{name:<35} {'—':>8}  {'—':<12}  {'—':<12}  {'—':>8}  ❌ empty")
        continue
    cols    = [c for c in KEY_COLS.get(name, []) if c in df.columns]
    missing = df[cols].isnull().mean().max() * 100 if cols else 0.0
    status  = "✅" if missing < 5 else "⚠️"
    print(
        f"{name:<35} {len(df):>8,}  "
        f"{str(df.index.min())[:10]:<12}  "
        f"{str(df.index.max())[:10]:<12}  "
        f"{missing:>7.1f}%  {status}"
    )

Dataset                                 Rows  From          To             Missing  Status
------------------------------------------------------------------------------------------
entso_unavailability                  22,100  2017-04-21    2026-03-31        0.0%  ✅
elexon_da_prices                       2,983  2018-01-01    2026-04-05        0.0%  ✅
elexon_mid_halfhourly                142,111  2018-01-01    2026-04-01        0.0%  ✅
neso_historic_demand                 143,952  2018-01-01    2026-03-18        0.0%  ✅
rte_generation                       283,484  2018-01-01    2026-01-31       50.0%  ⚠️
fr_temperature                         3,013  2018-01-01    2026-04-01        0.0%  ✅
ttf_spot                               3,011  2018-01-02    2026-03-31        0.0%  ✅
de_wind_generation                    72,288  2018-01-01    2026-03-31        0.0%  ✅
fr_da_price                           91,638  2017-12-31    2026-04-01        0.0%  ✅


**Note:**
rte_generation — 50% missing. RTE publishes at 30-minute intervals so every other row is empty. 

---
## 5. Stage 0 — Reduced-Form IFA Flow Regression

Estimated once here; used throughout the project for position sizing.

**Specification (estimated separately for IFA1 and IFA2):**
```
ifa_flow_mw(t) = α
               + β1 × fr_nuclear_actual_mw(t)   ← treatment channel
               + β2 × fr_net_exports_mw(t)       ← FR supply control
               + β3 × demand_actual_mw(t)        ← GB demand control
               + season_fe + hour_fe
               + u(t)                            ← HAC SE, 24 lags
```

- **IFA1:** full 2018–2026 sample
- **IFA2:** 2020–2026 only (implicit coupling commenced 2020)

**Output:** β1 — MW of IFA flow per MW of nuclear availability.
Converts an outage of size X MW into expected IFA import reduction.

In [13]:
def run_ifa_flow_regression(
    neso_df: pd.DataFrame,
    rte_df:  pd.DataFrame,
    interconnector: str = "ifa1",
    start: str = "2018-01-01",
) -> tuple:
    """
    OLS regression of IFA flow on French nuclear availability.
    Returns (statsmodels RegressionResults, model DataFrame).
    """
    flow_col = f"{interconnector}_flow_mw"

    neso_h = neso_df[[flow_col, "demand_actual_mw", "wind_actual_mw"]].resample("h").mean()
    rte_h  = rte_df[["fr_nuclear_actual_mw", "fr_net_exports_mw"]].resample("h").mean()

    df = neso_h.join(rte_h, how="inner").dropna()
    df = df[df.index >= pd.Timestamp(start, tz="UTC")]

    df["hour"]   = df.index.hour
    df["season"] = df.index.month.map(
        {12: 0, 1: 0, 2: 0,
          3: 1, 4: 1, 5: 1,
          6: 2, 7: 2, 8: 2,
          9: 3, 10: 3, 11: 3}
    )

    features = ["fr_nuclear_actual_mw", "fr_net_exports_mw", "demand_actual_mw"]
    X = pd.concat([
        df[features],
        pd.get_dummies(df["season"], prefix="season", drop_first=True),
        pd.get_dummies(df["hour"],   prefix="hour",   drop_first=True),
    ], axis=1)
    X = sm.add_constant(X)

    result = sm.OLS(df[flow_col], X.astype(float)).fit(
        cov_type="HAC", cov_kwds={"maxlags": 24}
    )
    return result, df

In [14]:
res_ifa1 = res_ifa2 = None

if neso_df.empty or rte_df.empty:
    print("Cannot run regression — NESO or RTE data missing.")
else:
    if "ifa1_flow_mw" in neso_df.columns:
        print("=== IFA1 Flow Regression (2018–2026) ===")
        res_ifa1, df_ifa1 = run_ifa_flow_regression(
            neso_df, rte_df, interconnector="ifa1", start="2018-01-01"
        )
        b1 = res_ifa1.params["fr_nuclear_actual_mw"] * 1000
        p1 = res_ifa1.pvalues["fr_nuclear_actual_mw"]
        print(f"  N = {res_ifa1.nobs:.0f}  |  R² = {res_ifa1.rsquared:.3f}")
        print(f"  β_nuclear = {b1:.1f} MW flow per GW nuclear  (p={p1:.3f})")

    if "ifa2_flow_mw" in neso_df.columns:
        print("\n=== IFA2 Flow Regression (2020–2026) ===")
        res_ifa2, df_ifa2 = run_ifa_flow_regression(
            neso_df, rte_df, interconnector="ifa2", start="2020-01-01"
        )
        b2 = res_ifa2.params["fr_nuclear_actual_mw"] * 1000
        p2 = res_ifa2.pvalues["fr_nuclear_actual_mw"]
        print(f"  N = {res_ifa2.nobs:.0f}  |  R² = {res_ifa2.rsquared:.3f}")
        print(f"  β_nuclear = {b2:.1f} MW flow per GW nuclear  (p={p2:.3f})")

    if res_ifa1 or res_ifa2:
        print("\nInterpretation: a 1,000 MW unplanned outage implies")
        if res_ifa1:
            mw1 = res_ifa1.params["fr_nuclear_actual_mw"] * 1000
            print(f"  IFA1 reduction ≈ {mw1:.0f} MW")
        if res_ifa2:
            mw2 = res_ifa2.params["fr_nuclear_actual_mw"] * 1000
            print(f"  IFA2 reduction ≈ {mw2:.0f} MW")
        if res_ifa1 and res_ifa2:
            combined = mw1 + mw2
            print(f"  Combined        ≈ {combined:.0f} MW  ({combined/30:.1f}% of 3 GW capacity)")

=== IFA1 Flow Regression (2018–2026) ===
  N = 70855  |  R² = 0.419
  β_nuclear = 25.4 MW flow per GW nuclear  (p=0.000)

=== IFA2 Flow Regression (2020–2026) ===
  N = 53339  |  R² = 0.382
  β_nuclear = 19.3 MW flow per GW nuclear  (p=0.000)

Interpretation: a 1,000 MW unplanned outage implies
  IFA1 reduction ≈ 25 MW
  IFA2 reduction ≈ 19 MW
  Combined        ≈ 45 MW  (1.5% of 3 GW capacity)


In [15]:
# Save regression coefficients
coef_records = []
for model_name, result in [("IFA1", res_ifa1), ("IFA2", res_ifa2)]:
    if result is None:
        continue
    for param in ["fr_nuclear_actual_mw", "fr_net_exports_mw", "demand_actual_mw", "const"]:
        if param in result.params:
            coef_records.append({
                "model":     model_name,
                "parameter": param,
                "coef":      result.params[param],
                "se":        result.bse[param],
                "pvalue":    result.pvalues[param],
                "nobs":      int(result.nobs),
                "r2":        result.rsquared,
            })

if coef_records:
    coef_df = pd.DataFrame(coef_records)
    coef_df.to_parquet(DATA_DIR / "ifa_flow_model.parquet", index=False)
    print("Saved ifa_flow_model.parquet\n")
    print(coef_df.to_string(index=False))
else:
    print("No regression results to save — check NESO and RTE data.")

Saved ifa_flow_model.parquet

model            parameter     coef    se  pvalue  nobs   r2
 IFA1 fr_nuclear_actual_mw     0.03  0.00    0.00 70855 0.42
 IFA1    fr_net_exports_mw    -0.09  0.00    0.00 70855 0.42
 IFA1     demand_actual_mw     0.03  0.00    0.00 70855 0.42
 IFA1                const -1646.40 98.47    0.00 70855 0.42
 IFA2 fr_nuclear_actual_mw     0.02  0.00    0.00 53339 0.38
 IFA2    fr_net_exports_mw    -0.05  0.00    0.00 53339 0.38
 IFA2     demand_actual_mw    -0.00  0.00    0.62 53339 0.38
 IFA2                const  -989.49 90.38    0.00 53339 0.38


---
## ✅ Notebook 01 Complete

All data cached to `data/`. Proceed to **Notebook 02b** — ID Market Pre-Gate.

**Note on GB intraday prices:** EPEX GB hourly ID settlement prices are not
publicly available post-Brexit. `elexon_mid_halfhourly` is used as the
intraday proxy in Notebook 02b. This constraint is pre-registered.
Results are stated at 1 MW and not extrapolated.

**To re-fetch any dataset:**
```python
# Single dataset
(DATA_DIR / "entso_unavailability.parquet").unlink(missing_ok=True)

# All datasets
for f in DATA_DIR.glob("*.parquet"): f.unlink()
```